In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

pd.set_option("display.max_columns", 100)

TARGET = "Will_Buy_EV"
ID_COL = "id"

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sub = pd.read_csv("sample_submission.csv")

print("train:", train.shape, "| test:", test.shape)

train: (668665, 15) | test: (286571, 14)


In [15]:
train.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [16]:
train.dtypes

id                               int64
Age                              int64
Annual_Income_USD              float64
Daily_Commute_km               float64
Number_of_Cars_Owned             int64
Charging_Stations_Near_Home      int64
Charging_Stations_Near_Work      int64
Environmental_Concern_Level    float64
Gender                          object
City_Type                       object
Current_Car_Type                object
Home_Charging_Possible          object
Subsidy_Available               object
Range_Anxiety_Level             object
Will_Buy_EV                     object
dtype: object

In [17]:
train.isna().sum()

id                             0
Age                            0
Annual_Income_USD              0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Environmental_Concern_Level    0
Gender                         0
City_Type                      0
Current_Car_Type               0
Home_Charging_Possible         0
Subsidy_Available              0
Range_Anxiety_Level            0
Will_Buy_EV                    0
dtype: int64

In [18]:
train[TARGET].value_counts(normalize=True)

Will_Buy_EV
No     0.825355
Yes    0.174645
Name: proportion, dtype: float64

In [19]:
train.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,668665.0,NaN,NaN,NaN,334332.0,193027.103211,0.0,167166.0,334332.0,501498.0,668664.0
Age,668665.0,NaN,NaN,NaN,47.039171,12.875448,25.0,36.0,47.0,58.0,69.0
Annual_Income_USD,668665.0,NaN,NaN,NaN,84769.266989,28648.029042,30000.0,67376.0,84880.0,102753.0,188549.0
Daily_Commute_km,668665.0,NaN,NaN,NaN,32.158298,18.730474,5.0,17.2,33.6,47.4,98.7
Number_of_Cars_Owned,668665.0,NaN,NaN,NaN,1.712626,0.729275,1.0,1.0,2.0,2.0,4.0
Charging_Stations_Near_Home,668665.0,NaN,NaN,NaN,4.960408,3.926843,0.0,2.0,4.0,7.0,14.0
Charging_Stations_Near_Work,668665.0,NaN,NaN,NaN,7.176314,5.186627,0.0,3.0,6.0,10.0,19.0
Environmental_Concern_Level,668665.0,NaN,NaN,NaN,2.935477,1.429119,1.0,2.0,3.0,4.0,5.0
Gender,668665,3,Male,367954,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City_Type,668665,3,Urban,289305,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
y = train[TARGET].map({"Yes": 1, "No": 0})
X = train.drop(columns=[TARGET, ID_COL])
X_test = test.drop(columns=[ID_COL])
test_ids = test[ID_COL]

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical:", cat_cols)
print("Numeric:", num_cols)

Categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
Numeric: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']


In [20]:
range_anxiety_map = {"Low": 0, "Medium": 1, "High": 2}
X["Range_Anxiety_Level"] = X["Range_Anxiety_Level"].map(range_anxiety_map)
X_test["Range_Anxiety_Level"] = X_test["Range_Anxiety_Level"].map(range_anxiety_map)

cat_cols = [c for c in cat_cols if c != "Range_Anxiety_Level"]

for df in [X, X_test]:
    df["Total_Charging_Stations"] = df["Charging_Stations_Near_Home"] + df["Charging_Stations_Near_Work"]
    df["Commute_per_Car"] = df["Daily_Commute_km"] / df["Number_of_Cars_Owned"]
    df["Income_per_Age"] = df["Annual_Income_USD"] / df["Age"]

if cat_cols:
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    combined = pd.concat([X[cat_cols], X_test[cat_cols]], axis=0)
    enc.fit(combined)
    X[cat_cols] = enc.transform(X[cat_cols])
    X_test[cat_cols] = enc.transform(X_test[cat_cols])

print("Kalan kategorikler:", cat_cols)
print(X.dtypes)

Kalan kategorikler: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available']
Age                              int64
Annual_Income_USD              float64
Daily_Commute_km               float64
Number_of_Cars_Owned             int64
Charging_Stations_Near_Home      int64
Charging_Stations_Near_Work      int64
Environmental_Concern_Level    float64
Gender                         float64
City_Type                      float64
Current_Car_Type               float64
Home_Charging_Possible         float64
Subsidy_Available              float64
Range_Anxiety_Level            float64
Total_Charging_Stations          int64
Commute_per_Car                float64
Income_per_Age                 float64
dtype: object


In [21]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
models = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = HistGradientBoostingClassifier(
        max_iter=500,
        learning_rate=0.05,
        max_depth=None,
        max_leaf_nodes=63,
        min_samples_leaf=30,
        l2_regularization=0.1,
        early_stopping=True,
        n_iter_no_change=30,
        validation_fraction=0.1,
        random_state=42,
    )
    model.fit(X_tr, y_tr)

    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / N_SPLITS
    models.append(model)

    fold_auc = roc_auc_score(y_val, oof[val_idx])
    print(f"Fold {fold+1} AUC: {fold_auc:.5f}")

overall_auc = roc_auc_score(y, oof)
print(f"\nOverall OOF AUC: {overall_auc:.5f}")

Fold 1 AUC: 0.93839
Fold 2 AUC: 0.93944
Fold 3 AUC: 0.94075
Fold 4 AUC: 0.94042
Fold 5 AUC: 0.93942

Overall OOF AUC: 0.93967


In [22]:
submission = pd.DataFrame({ID_COL: test_ids, TARGET: test_preds})
submission.to_csv("submission.csv", index=False)
submission.head()

,id,Will_Buy_EV
0,668665,0.007981
1,668666,0.021992
2,668667,0.004412
3,668668,0.003304
4,668669,0.015687


In [23]:
X_cb = train.drop(columns=[TARGET, ID_COL]).copy()
X_test_cb = test.drop(columns=[ID_COL]).copy()

for df in [X_cb, X_test_cb]:
    df["Total_Charging_Stations"] = df["Charging_Stations_Near_Home"] + df["Charging_Stations_Near_Work"]
    df["Commute_per_Car"] = df["Daily_Commute_km"] / df["Number_of_Cars_Owned"]
    df["Income_per_Age"] = df["Annual_Income_USD"] / df["Age"]

cat_cols_cb = X_cb.select_dtypes(include=["object"]).columns.tolist()
X_cb[cat_cols_cb] = X_cb[cat_cols_cb].fillna("Missing")
X_test_cb[cat_cols_cb] = X_test_cb[cat_cols_cb].fillna("Missing")

print("CatBoost categorical cols:", cat_cols_cb)

CatBoost categorical cols: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


In [24]:
from catboost import CatBoostClassifier

cat_features_idx = [X_cb.columns.get_loc(c) for c in cat_cols_cb]

oof_cb = np.zeros(len(X_cb))
test_preds_cb = np.zeros(len(X_test_cb))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cb, y)):
    X_tr, X_val = X_cb.iloc[tr_idx], X_cb.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model_cb = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3,
        eval_metric="AUC",
        cat_features=cat_features_idx,
        random_seed=42,
        verbose=False,
        early_stopping_rounds=100,
    )
    model_cb.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    oof_cb[val_idx] = model_cb.predict_proba(X_val)[:, 1]
    test_preds_cb += model_cb.predict_proba(X_test_cb)[:, 1] / N_SPLITS

    print(f"Fold {fold+1} CatBoost AUC: {roc_auc_score(y_val, oof_cb[val_idx]):.5f}")

print(f"\nCatBoost Overall OOF AUC: {roc_auc_score(y, oof_cb):.5f}")


Fold 1 CatBoost AUC: 0.94039
Fold 2 CatBoost AUC: 0.94124
Fold 3 CatBoost AUC: 0.94248
Fold 4 CatBoost AUC: 0.94214
Fold 5 CatBoost AUC: 0.94148

CatBoost Overall OOF AUC: 0.94154


In [25]:
blend_oof = 0.5 * oof + 0.5 * oof_cb
print("Blend OOF AUC:", roc_auc_score(y, blend_oof))

blend_test = 0.5 * test_preds + 0.5 * test_preds_cb
submission_blend = pd.DataFrame({ID_COL: test_ids, TARGET: blend_test})
submission_blend.to_csv("submission_blend.csv", index=False)
submission_blend.head()

Blend OOF AUC: 0.9411607678544122


,id,Will_Buy_EV
0,668665,0.008171
1,668666,0.023388
2,668667,0.004461
3,668668,0.002986
4,668669,0.019360


In [26]:
from scipy.stats import rankdata

# Rank-based blend
oof_rank_blend = (rankdata(oof) + rankdata(oof_cb)) / 2
print("Rank blend OOF AUC:", roc_auc_score(y, oof_rank_blend))

# Farklı ağırlık kombinasyonlarını dene
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    blend = w * oof + (1 - w) * oof_cb
    auc = roc_auc_score(y, blend)
    print(f"w={w}: weighted blend OOF AUC = {auc:.5f}")

print(f"\nHistGBM tek başına: {roc_auc_score(y, oof):.5f}")
print(f"CatBoost tek başına: {roc_auc_score(y, oof_cb):.5f}")

Rank blend OOF AUC: 0.9412236075207205
w=0.3: weighted blend OOF AUC = 0.94145
w=0.4: weighted blend OOF AUC = 0.94133
w=0.5: weighted blend OOF AUC = 0.94116
w=0.6: weighted blend OOF AUC = 0.94095
w=0.7: weighted blend OOF AUC = 0.94070

HistGBM tek başına: 0.93967
CatBoost tek başına: 0.94154


In [27]:
submission_final = pd.DataFrame({ID_COL: test_ids, TARGET: test_preds_cb})
submission_final.to_csv("submission_final.csv", index=False)
submission_final.head()

,id,Will_Buy_EV
0,668665,0.008360
1,668666,0.024783
2,668667,0.004509
3,668668,0.002669
4,668669,0.023034


In [30]:
from sklearn.linear_model import LogisticRegression

# Meta-feature: iki modelin OOF tahminleri
meta_X = np.column_stack([oof, oof_cb])
meta_X_test = np.column_stack([test_preds, test_preds_cb])

meta_oof = np.zeros(len(y))
meta_test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(meta_X, y)):
    meta_model = LogisticRegression()
    meta_model.fit(meta_X[tr_idx], y.iloc[tr_idx])
    meta_oof[val_idx] = meta_model.predict_proba(meta_X[val_idx])[:, 1]
    meta_test_preds += meta_model.predict_proba(meta_X_test)[:, 1] / N_SPLITS

print("Stacking OOF AUC:", roc_auc_score(y, meta_oof))

Stacking OOF AUC: 0.9414960527493965
